<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_16_Controlled_Original_TRACE_20B_Replay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from pathlib import Path
import os, json, hashlib, shutil, sys, tempfile, ast, contextlib, io, collections
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

MYDRIVE = Path(os.environ.get('ICHI2027_MYDRIVE', '/content/drive/MyDrive')).expanduser().resolve()
assert MYDRIVE.is_dir(), 'Mount your existing Google Drive before running Notebook 16.'
PROJECT = MYDRIVE / 'NeuroFHIR_ICHI2027'
OUT = PROJECT / 'phase16_outputs'
PRIVATE = OUT / '_PRIVATE_DO_NOT_SHARE'
PRIVATE.mkdir(parents=True, exist_ok=True)
SHAREABLE = OUT / 'ICHI2027_Phase16_SHAREABLE.json'
RUNROOT = Path(tempfile.mkdtemp(prefix='ichi2027_phase16_'))

def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''):h.update(block)
    return h.hexdigest()

def src(c):
    s=c.get('source','')
    return ''.join(s) if isinstance(s,list) else str(s)

def original_fingerprint(parts):
    return hashlib.sha256('\n# CELL BREAK\n'.join(s.replace('\r\n','\n').strip() for s in parts).encode('utf-8')).hexdigest()

def contained_file(root,relative):
    if not isinstance(relative,str) or not relative:return None
    root=Path(root).resolve(); p=(root/relative).resolve()
    return p if p.is_relative_to(root) and p.is_file() else None

phase15_path=Path(os.environ.get('ICHI2027_PHASE15_JSON',str(PROJECT/'phase15_outputs'/'ICHI2027_Phase15_SHAREABLE.json')))
if not phase15_path.is_file():
    try:
        from google.colab import files
        selected=files.upload()
        if 'ICHI2027_Phase15_SHAREABLE.json' in selected:
            phase15_path=RUNROOT/'ICHI2027_Phase15_SHAREABLE.json'
            phase15_path.write_bytes(selected['ICHI2027_Phase15_SHAREABLE.json'])
    except ImportError: pass
assert phase15_path.is_file(), 'Phase15 shareable JSON missing. Restore it to phase15_outputs on Drive.'
phase15=json.loads(phase15_path.read_text(encoding='utf-8'))
assert phase15.get('phase')=='15_original_20b_method_extraction_and_authentic_input_preflight'
assert phase15.get('gates',{}).get('ready_for_manual_review_before_original_20b_replay') is True
assert phase15.get('gates',{}).get('phase09_longitudinal_source_hash_reconciled') is True
private15_path=Path(os.environ.get('ICHI2027_PHASE15_PRIVATE',str(PROJECT/'phase15_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase15_PRIVATE_METHOD_AND_INPUT_PREFLIGHT.json')))
assert private15_path.is_file(), ('Phase15 PRIVATE preflight manifest missing on Drive. Rerun Notebook15 '
    'on the same Drive; do not share the private manifest publicly.')
private15=json.loads(private15_path.read_text(encoding='utf-8'))
assert private15.get('run_utc')==phase15.get('run_utc'),'Phase15 public/private manifests are from different runs.'
assert private15.get('phase14_shareable_sha256')==phase15.get('phase14_shareable_sha256')
print('Phase15 confirmed; original private hashes available. Original code has not run yet.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Phase15 confirmed; original private hashes available. Original code has not run yet.


In [5]:
ORIGINAL_20B_CELLS = [
 "\nfrom pathlib import Path\nimport json\nimport hashlib\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nwarnings.filterwarnings(\"ignore\")\n\nSEED = 2027\nrng = np.random.default_rng(SEED)\n\nOUTDIR = Path(\"trace_20b_real_corrected_outputs\")\nOUTDIR.mkdir(exist_ok=True)\n\nprint(\"Output directory:\", OUTDIR.resolve())\n",
 "\nEXPECTED_LONG = \"TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv\"\nEXPECTED_REP = \"TRACE_ADNI1_repeat_pairs_REAL.csv\"\n\nSEARCH_DIRS = [Path(\"/content\"), Path.cwd(), Path(\"/mnt/data\")]\n\ndef find_file(expected_name):\n    stem = Path(expected_name).stem\n    suffix = Path(expected_name).suffix.lower()\n\n    for d in SEARCH_DIRS:\n        p = d / expected_name\n        if p.exists():\n            return p\n\n    candidates = []\n    for d in SEARCH_DIRS:\n        if d.exists():\n            candidates.extend(\n                p for p in d.glob(f\"{stem}*{suffix}\") if p.is_file()\n            )\n\n    if not candidates:\n        return None\n\n    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]\n\nLONGITUDINAL_PATH = find_file(EXPECTED_LONG)\nREPEAT_PATH = find_file(EXPECTED_REP)\n\nif LONGITUDINAL_PATH is None or REPEAT_PATH is None:\n    missing = []\n    if LONGITUDINAL_PATH is None:\n        missing.append(EXPECTED_LONG)\n    if REPEAT_PATH is None:\n        missing.append(EXPECTED_REP)\n    raise FileNotFoundError(\n        \"Missing required file(s): \" + \", \".join(missing) +\n        \". Upload both using the Colab left Files pane, then rerun this cell.\"\n    )\n\nlong_df = pd.read_csv(LONGITUDINAL_PATH)\nrep_df = pd.read_csv(REPEAT_PATH)\n\nprint(\"Using longitudinal file:\", LONGITUDINAL_PATH)\nprint(\"Using repeat-pair file:\", REPEAT_PATH)\nprint(\"Longitudinal shape:\", long_df.shape)\nprint(\"Repeat-pair shape:\", rep_df.shape)\n",
 "\nFEATURES = [\n    \"LHIPPOC\",\n    \"RHIPPOC\",\n    \"LENTORHIN\",\n    \"RENTORHIN\",\n    \"LMIDTEMP\",\n    \"RMIDTEMP\",\n    \"VENTRICLES\",\n    \"ICV\",\n]\nBIO_FEATURES = [f for f in FEATURES if f != \"ICV\"]\n\nACQ_TYPE = \"SAME_DAY_DIFFERENT_ACQUISITION\"\nREPROC_TYPE = \"SAME_SOURCE_REPROCESSING\"\n\nrequired_long = (\n    [\"RID\", \"LABEL\", \"BL_OVERALLQC\", \"M12_OVERALLQC\",\n     \"BL_TEMPQC\", \"M12_TEMPQC\", \"BL_VENTQC\", \"M12_VENTQC\"]\n    + [f\"BL_{f}\" for f in FEATURES]\n    + [f\"M12_{f}\" for f in FEATURES]\n)\n\nrequired_rep = (\n    [\"RID\", \"REPEAT_TYPE\", \"QC_1\", \"QC_2\",\n     \"TEMPQC_1\", \"TEMPQC_2\", \"VENTQC_1\", \"VENTQC_2\"]\n    + [f\"PCT_DIFF_{f}\" for f in FEATURES]\n)\n\nmissing_long = [c for c in required_long if c not in long_df.columns]\nmissing_rep = [c for c in required_rep if c not in rep_df.columns]\n\nif missing_long:\n    raise ValueError(f\"Missing longitudinal columns: {missing_long}\")\nif missing_rep:\n    raise ValueError(f\"Missing repeat columns: {missing_rep}\")\n\nacq = rep_df.loc[rep_df[\"REPEAT_TYPE\"] == ACQ_TYPE].copy()\nreproc = rep_df.loc[rep_df[\"REPEAT_TYPE\"] == REPROC_TYPE].copy()\n\nprint(\"Longitudinal subjects:\", long_df[\"RID\"].nunique())\nprint(\"Same-day acquisition pairs:\", len(acq))\nprint(\"Same-day acquisition subjects:\", acq[\"RID\"].nunique())\nprint(\"Repeated-processing pairs:\", len(reproc))\nprint(\"Repeated-processing subjects:\", reproc[\"RID\"].nunique())\n",
 "\ndef make_subject_folds(subjects, n_splits=5, local_rng=None):\n    subjects = np.array(subjects)\n    perm = subjects.copy()\n    local_rng.shuffle(perm)\n    return np.array_split(perm, n_splits)\n\nCV_REPEATS = 25\nN_SPLITS = 5\n\nunique_subjects = np.array(sorted(acq[\"RID\"].dropna().unique()))\ncv_rows = []\n\nfor repeat_idx in range(CV_REPEATS):\n    local_rng = np.random.default_rng(SEED + repeat_idx)\n    folds = make_subject_folds(unique_subjects, n_splits=N_SPLITS, local_rng=local_rng)\n\n    for fold_idx, test_subjects in enumerate(folds):\n        test_subjects = set(test_subjects.tolist())\n        train = acq.loc[~acq[\"RID\"].isin(test_subjects)]\n        test = acq.loc[acq[\"RID\"].isin(test_subjects)]\n\n        for f in FEATURES:\n            tr = pd.to_numeric(train[f\"PCT_DIFF_{f}\"], errors=\"coerce\").dropna().to_numpy()\n            te = pd.to_numeric(test[f\"PCT_DIFF_{f}\"], errors=\"coerce\").dropna().to_numpy()\n\n            if len(tr) == 0 or len(te) == 0:\n                continue\n\n            for q in [90, 95]:\n                threshold = float(np.percentile(tr, q))\n                cv_rows.append({\n                    \"repeat\": repeat_idx,\n                    \"fold\": fold_idx,\n                    \"feature\": f,\n                    \"quantile\": q,\n                    \"train_pairs\": len(tr),\n                    \"test_pairs\": len(te),\n                    \"threshold_pct\": threshold,\n                    \"heldout_coverage_pct\": 100.0 * float(np.mean(te <= threshold)),\n                    \"heldout_exceedance_pct\": 100.0 * float(np.mean(te > threshold)),\n                })\n\ncv = pd.DataFrame(cv_rows)\n\ncv_summary = (\n    cv.groupby([\"feature\", \"quantile\"])\n      .agg(\n          folds=(\"heldout_coverage_pct\", \"size\"),\n          mean_coverage_pct=(\"heldout_coverage_pct\", \"mean\"),\n          median_coverage_pct=(\"heldout_coverage_pct\", \"median\"),\n          sd_coverage_pct=(\"heldout_coverage_pct\", \"std\"),\n          p10_coverage_pct=(\"heldout_coverage_pct\", lambda x: np.percentile(x, 10)),\n          p90_coverage_pct=(\"heldout_coverage_pct\", lambda x: np.percentile(x, 90)),\n          median_threshold_pct=(\"threshold_pct\", \"median\"),\n      )\n      .reset_index()\n)\n\ncv_summary.round(2)\n",
 "\ndef cluster_bootstrap_quantile(df, feature, q, n_boot=2000, seed=SEED):\n    local_rng = np.random.default_rng(seed)\n    subjects = np.array(sorted(df[\"RID\"].dropna().unique()))\n    out = []\n\n    grouped = {rid: g for rid, g in df.groupby(\"RID\")}\n\n    for _ in range(n_boot):\n        sampled = local_rng.choice(subjects, size=len(subjects), replace=True)\n        vals = []\n        for rid in sampled:\n            z = pd.to_numeric(\n                grouped[rid][f\"PCT_DIFF_{feature}\"],\n                errors=\"coerce\"\n            ).dropna().to_numpy()\n            vals.extend(z.tolist())\n\n        if len(vals):\n            out.append(np.percentile(vals, q))\n\n    return np.asarray(out, dtype=float)\n\nthreshold_rows = []\n\nfor f in FEATURES:\n    vals = pd.to_numeric(acq[f\"PCT_DIFF_{f}\"], errors=\"coerce\").dropna().to_numpy()\n\n    boot90 = cluster_bootstrap_quantile(acq, f, 90, n_boot=2000, seed=SEED)\n    boot95 = cluster_bootstrap_quantile(acq, f, 95, n_boot=2000, seed=SEED + 100)\n\n    threshold_rows.append({\n        \"feature\": f,\n        \"n_pairs\": len(vals),\n        \"n_subjects\": acq[\"RID\"].nunique(),\n        \"q50_pct\": float(np.percentile(vals, 50)),\n        \"q90_pct\": float(np.percentile(vals, 90)),\n        \"q90_boot_ci_low\": float(np.percentile(boot90, 2.5)),\n        \"q90_boot_ci_high\": float(np.percentile(boot90, 97.5)),\n        \"q95_pct\": float(np.percentile(vals, 95)),\n        \"q95_boot_ci_low\": float(np.percentile(boot95, 2.5)),\n        \"q95_boot_ci_high\": float(np.percentile(boot95, 97.5)),\n    })\n\nthresholds = pd.DataFrame(threshold_rows)\nthresholds.round(4)\n",
 "\nreproc_rows = []\n\nfor f in FEATURES:\n    vals = pd.to_numeric(reproc[f\"PCT_DIFF_{f}\"], errors=\"coerce\").dropna()\n    q90 = float(thresholds.loc[thresholds[\"feature\"] == f, \"q90_pct\"].iloc[0])\n    q95 = float(thresholds.loc[thresholds[\"feature\"] == f, \"q95_pct\"].iloc[0])\n\n    reproc_rows.append({\n        \"feature\": f,\n        \"n_reprocessing_pairs\": int(len(vals)),\n        \"median_reprocessing_pct\": float(vals.median()) if len(vals) else np.nan,\n        \"coverage_under_acq_q90_pct\": 100.0 * float((vals <= q90).mean()) if len(vals) else np.nan,\n        \"coverage_under_acq_q95_pct\": 100.0 * float((vals <= q95).mean()) if len(vals) else np.nan,\n    })\n\nreproc_sensitivity = pd.DataFrame(reproc_rows)\nreproc_sensitivity.round(2)\n",
 "\ngate_df = long_df.copy()\n\nfor f in FEATURES:\n    bl = pd.to_numeric(gate_df[f\"BL_{f}\"], errors=\"coerce\")\n    m12 = pd.to_numeric(gate_df[f\"M12_{f}\"], errors=\"coerce\")\n    pct = 100.0 * np.abs(m12 - bl) / np.abs(bl)\n    pct = pct.replace([np.inf, -np.inf], np.nan)\n\n    q90 = float(thresholds.loc[thresholds[\"feature\"] == f, \"q90_pct\"].iloc[0])\n    q95 = float(thresholds.loc[thresholds[\"feature\"] == f, \"q95_pct\"].iloc[0])\n\n    gate_df[f\"{f}_ABS_LONG_PCT\"] = pct\n    gate_df[f\"{f}_TECH_Q90\"] = q90\n    gate_df[f\"{f}_TECH_Q95\"] = q95\n    gate_df[f\"{f}_SIGNAL_TO_TECH_Q90\"] = pct / q90\n    gate_df[f\"{f}_EXCEEDS_Q90\"] = pct > q90\n    gate_df[f\"{f}_EXCEEDS_Q95\"] = pct > q95\n\nprint(\"Longitudinal subjects:\", gate_df[\"RID\"].nunique())\n",
 "\ngate_df[\"ICV_HARD_REVIEW\"] = gate_df[\"ICV_EXCEEDS_Q95\"].fillna(True)\n\ngate_df[\"TEMP_HARD_REVIEW\"] = (\n    gate_df[\"BL_TEMPQC\"].astype(str).eq(\"Fail\")\n    | gate_df[\"M12_TEMPQC\"].astype(str).eq(\"Fail\")\n)\n\ngate_df[\"VENT_HARD_REVIEW\"] = (\n    gate_df[\"BL_VENTQC\"].astype(str).eq(\"Fail\")\n    | gate_df[\"M12_VENTQC\"].astype(str).eq(\"Fail\")\n)\n\ngate_df[\"OVERALL_QC_BOTH_PASS\"] = (\n    gate_df[\"BL_OVERALLQC\"].astype(str).eq(\"Pass\")\n    & gate_df[\"M12_OVERALLQC\"].astype(str).eq(\"Pass\")\n)\n\ngate_df[\"N_UNIQUE_HARD_ROOT_CAUSES\"] = (\n    gate_df[\n        [\"ICV_HARD_REVIEW\", \"TEMP_HARD_REVIEW\", \"VENT_HARD_REVIEW\"]\n    ].astype(int).sum(axis=1)\n)\n\nroot_cause_summary = pd.DataFrame({\n    \"root_cause\": [\n        \"ICV_HARD_REVIEW\",\n        \"TEMP_HARD_REVIEW\",\n        \"VENT_HARD_REVIEW\",\n        \"ANY_HARD_REVIEW\",\n    ],\n    \"n\": [\n        int(gate_df[\"ICV_HARD_REVIEW\"].sum()),\n        int(gate_df[\"TEMP_HARD_REVIEW\"].sum()),\n        int(gate_df[\"VENT_HARD_REVIEW\"].sum()),\n        int((gate_df[\"N_UNIQUE_HARD_ROOT_CAUSES\"] > 0).sum()),\n    ],\n})\n\nroot_cause_summary[\"pct\"] = 100.0 * root_cause_summary[\"n\"] / len(gate_df)\nroot_cause_summary.round(1)\n",
 "\ndef feature_hard_review(df, feature):\n    hard = df[\"ICV_HARD_REVIEW\"].copy()\n\n    if feature in {\"LMIDTEMP\", \"RMIDTEMP\"}:\n        hard = hard | df[\"TEMP_HARD_REVIEW\"]\n\n    if feature == \"VENTRICLES\":\n        hard = hard | df[\"VENT_HARD_REVIEW\"]\n\n    return hard\n\nfor f in BIO_FEATURES:\n    hard = feature_hard_review(gate_df, f)\n    trusted = (\n        (~hard)\n        & gate_df[\"OVERALL_QC_BOTH_PASS\"]\n        & gate_df[f\"{f}_EXCEEDS_Q90\"].fillna(False)\n    )\n\n    state = np.where(\n        hard,\n        \"REVIEW_REQUIRED\",\n        np.where(trusted, \"TRUSTED\", \"CAUTION\")\n    )\n\n    gate_df[f\"{f}_TRACE_STATE\"] = state\n\nstate_rows = []\nfor f in BIO_FEATURES:\n    counts = gate_df[f\"{f}_TRACE_STATE\"].value_counts()\n    for state in [\"TRUSTED\", \"CAUTION\", \"REVIEW_REQUIRED\"]:\n        state_rows.append({\n            \"feature\": f,\n            \"state\": state,\n            \"n\": int(counts.get(state, 0)),\n            \"pct\": 100.0 * float(counts.get(state, 0)) / len(gate_df),\n        })\n\nfeature_state_summary = pd.DataFrame(state_rows)\n\nfeature_state_pivot = feature_state_summary.pivot(\n    index=\"feature\",\n    columns=\"state\",\n    values=\"pct\"\n).fillna(0)\n\nfeature_state_pivot.round(1)\n",
 "\nstate_cols = [f\"{f}_TRACE_STATE\" for f in BIO_FEATURES]\n\ngate_df[\"N_TRUSTED\"] = (gate_df[state_cols] == \"TRUSTED\").sum(axis=1)\ngate_df[\"N_CAUTION\"] = (gate_df[state_cols] == \"CAUTION\").sum(axis=1)\ngate_df[\"N_REVIEW_FEATURES\"] = (gate_df[state_cols] == \"REVIEW_REQUIRED\").sum(axis=1)\n\ndef package_state(df, trusted_min=4):\n    return np.where(\n        df[\"N_UNIQUE_HARD_ROOT_CAUSES\"] > 0,\n        \"REVIEW_REQUIRED\",\n        np.where(\n            df[\"N_TRUSTED\"] >= trusted_min,\n            \"TRUSTED\",\n            \"CAUTION\"\n        )\n    )\n\ngate_df[\"TRACE_PACKAGE_STATE\"] = package_state(gate_df, trusted_min=4)\n\nprint(gate_df[\"TRACE_PACKAGE_STATE\"].value_counts())\nprint()\nprint((100 * gate_df[\"TRACE_PACKAGE_STATE\"].value_counts(normalize=True)).round(1))\n",
 "\npackage_sensitivity_rows = []\n\nfor k in [3, 4, 5]:\n    s = pd.Series(package_state(gate_df, trusted_min=k))\n    counts = s.value_counts()\n\n    for state in [\"TRUSTED\", \"CAUTION\", \"REVIEW_REQUIRED\"]:\n        package_sensitivity_rows.append({\n            \"trusted_min_rule\": k,\n            \"state\": state,\n            \"n\": int(counts.get(state, 0)),\n            \"pct\": 100.0 * float(counts.get(state, 0)) / len(gate_df),\n        })\n\npackage_sensitivity = pd.DataFrame(package_sensitivity_rows)\npackage_sensitivity.pivot(\n    index=\"trusted_min_rule\",\n    columns=\"state\",\n    values=\"pct\"\n).fillna(0).round(1)\n",
 "\nq95_rows = []\n\nfor f in BIO_FEATURES:\n    hard = feature_hard_review(gate_df, f)\n    trusted_q95 = (\n        (~hard)\n        & gate_df[\"OVERALL_QC_BOTH_PASS\"]\n        & gate_df[f\"{f}_EXCEEDS_Q95\"].fillna(False)\n    )\n\n    q95_state = np.where(\n        hard,\n        \"REVIEW_REQUIRED\",\n        np.where(trusted_q95, \"TRUSTED\", \"CAUTION\")\n    )\n\n    counts = pd.Series(q95_state).value_counts()\n\n    for state in [\"TRUSTED\", \"CAUTION\", \"REVIEW_REQUIRED\"]:\n        q95_rows.append({\n            \"feature\": f,\n            \"state\": state,\n            \"n\": int(counts.get(state, 0)),\n            \"pct\": 100.0 * float(counts.get(state, 0)) / len(gate_df),\n        })\n\nq95_sensitivity = pd.DataFrame(q95_rows)\nq95_sensitivity.pivot(\n    index=\"feature\",\n    columns=\"state\",\n    values=\"pct\"\n).fillna(0).round(1)\n",
 "\noutcome_audit = pd.crosstab(\n    gate_df[\"TRACE_PACKAGE_STATE\"],\n    gate_df[\"LABEL\"],\n    margins=True\n)\n\noutcome_audit\n",
 "\n# Figure 1: repeated CV coverage for Q90 and Q95\nplot_cv = (\n    cv_summary.groupby(\"quantile\")[\"mean_coverage_pct\"]\n    .mean()\n    .reindex([90, 95])\n)\n\nfig, ax = plt.subplots(figsize=(6.5, 4.5))\nax.bar(plot_cv.index.astype(str), plot_cv.values)\nax.set_ylim(0, 100)\nax.set_ylabel(\"Mean held-out coverage (%)\")\nax.set_xlabel(\"Technical envelope quantile\")\nax.set_title(\"Repeated subject-level CV of technical envelopes\")\nplt.tight_layout()\nplt.savefig(OUTDIR / \"figure_cv_coverage_q90_q95.png\", dpi=300, bbox_inches=\"tight\")\nplt.show()\n",
 "\n# Figure 2: feature-level primary Q90 TRACE states\nplot_states = feature_state_pivot.reindex(\n    columns=[\"TRUSTED\", \"CAUTION\", \"REVIEW_REQUIRED\"]\n)\n\nax = plot_states.plot(kind=\"bar\", stacked=True, figsize=(11, 5))\nax.set_ylabel(\"Evidence items (%)\")\nax.set_xlabel(\"MRI biomarker\")\nax.set_title(\"Corrected real ADNI1 TRACE feature-level reliability states\")\nplt.xticks(rotation=45, ha=\"right\")\nplt.legend(title=\"TRACE state\")\nplt.tight_layout()\nplt.savefig(OUTDIR / \"figure_corrected_feature_trace_states.png\", dpi=300, bbox_inches=\"tight\")\nplt.show()\n",
 "\n# Figure 3: Q90 threshold with bootstrap interval\nfig, ax = plt.subplots(figsize=(11, 5))\nx = np.arange(len(thresholds))\n\nvals = thresholds[\"q90_pct\"].to_numpy()\nlower = vals - thresholds[\"q90_boot_ci_low\"].to_numpy()\nupper = thresholds[\"q90_boot_ci_high\"].to_numpy() - vals\n\nax.errorbar(\n    x,\n    vals,\n    yerr=np.vstack([lower, upper]),\n    fmt=\"o\",\n    capsize=4\n)\nax.set_xticks(x)\nax.set_xticklabels(thresholds[\"feature\"], rotation=45, ha=\"right\")\nax.set_ylabel(\"Absolute technical disagreement (%)\")\nax.set_xlabel(\"MRI biomarker\")\nax.set_title(\"Locked Q90 technical envelope with subject-bootstrap 95% CI\")\nplt.tight_layout()\nplt.savefig(OUTDIR / \"figure_q90_bootstrap_thresholds.png\", dpi=300, bbox_inches=\"tight\")\nplt.show()\n",
 "\ncv.to_csv(OUTDIR / \"table_repeated_cv_all_folds.csv\", index=False)\ncv_summary.to_csv(OUTDIR / \"table_repeated_cv_summary.csv\", index=False)\nthresholds.to_csv(OUTDIR / \"table_locked_thresholds_bootstrap.csv\", index=False)\nreproc_sensitivity.to_csv(OUTDIR / \"table_reprocessing_sensitivity.csv\", index=False)\nroot_cause_summary.to_csv(OUTDIR / \"table_unique_root_causes.csv\", index=False)\nfeature_state_summary.to_csv(OUTDIR / \"table_feature_trace_states_q90.csv\", index=False)\npackage_sensitivity.to_csv(OUTDIR / \"table_package_rule_sensitivity.csv\", index=False)\nq95_sensitivity.to_csv(OUTDIR / \"table_feature_trace_states_q95_sensitivity.csv\", index=False)\noutcome_audit.to_csv(OUTDIR / \"table_exploratory_outcome_audit.csv\")\n\nprint(\"Aggregate/publication-oriented outputs:\")\nfor p in sorted(OUTDIR.glob(\"*\")):\n    print(\" -\", p.name)\n",
 "\nPRIVATE_GATE_PATH = Path(\"TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE.csv\")\n\nprivate_cols = [\n    \"RID\",\n    \"LABEL\",\n    \"TRACE_PACKAGE_STATE\",\n    \"N_TRUSTED\",\n    \"N_CAUTION\",\n    \"N_REVIEW_FEATURES\",\n    \"N_UNIQUE_HARD_ROOT_CAUSES\",\n    \"ICV_HARD_REVIEW\",\n    \"TEMP_HARD_REVIEW\",\n    \"VENT_HARD_REVIEW\",\n    \"OVERALL_QC_BOTH_PASS\",\n]\n\nfor f in FEATURES:\n    for c in [\n        f\"BL_{f}\",\n        f\"M12_{f}\",\n        f\"{f}_ABS_LONG_PCT\",\n        f\"{f}_TECH_Q90\",\n        f\"{f}_TECH_Q95\",\n        f\"{f}_SIGNAL_TO_TECH_Q90\",\n    ]:\n        if c in gate_df.columns:\n            private_cols.append(c)\n\nfor f in BIO_FEATURES:\n    private_cols.append(f\"{f}_TRACE_STATE\")\n\nprivate_gate = gate_df[private_cols].copy()\nprivate_gate.to_csv(PRIVATE_GATE_PATH, index=False)\n\nprint(\"PRIVATE participant-level file saved:\")\nprint(PRIVATE_GATE_PATH.resolve())\nprint(\"Rows:\", len(private_gate))\nprint(\"DO NOT PUSH THIS FILE TO GITHUB.\")\n",
 "\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\nmanifest = {\n    \"notebook\": \"20B-REAL-CORRECTED\",\n    \"seed\": SEED,\n    \"inputs\": {\n        LONGITUDINAL_PATH.name: sha256_file(LONGITUDINAL_PATH),\n        REPEAT_PATH.name: sha256_file(REPEAT_PATH),\n    },\n    \"technical_reference\": {\n        \"same_day_acquisition_pairs\": int(len(acq)),\n        \"same_day_acquisition_subjects\": int(acq[\"RID\"].nunique()),\n        \"primary_threshold\": \"all-pair Q90 absolute percent disagreement\",\n        \"conservative_sensitivity_threshold\": \"all-pair Q95 absolute percent disagreement\",\n        \"threshold_uncertainty\": \"2000 subject-cluster bootstrap replicates\",\n        \"cross_validation\": f\"{CV_REPEATS}x repeated {N_SPLITS}-fold subject-level CV\",\n    },\n    \"longitudinal_subjects\": int(gate_df[\"RID\"].nunique()),\n    \"outcome_labels_used_for_gate_calibration\": False,\n    \"package_review_double_counts_bilateral_qc\": False,\n}\n\nwith open(OUTDIR / \"manifest.json\", \"w\") as f:\n    json.dump(manifest, f, indent=2)\n\nmanifest\n",
 "\nprint(\"============================================================\")\nprint(\"20B-REAL-CORRECTED READINESS SUMMARY\")\nprint(\"============================================================\")\n\nprint(f\"Primary same-day acquisition pairs: {len(acq)}\")\nprint(f\"Primary same-day acquisition subjects: {acq['RID'].nunique()}\")\nprint(f\"Longitudinal subjects gated: {gate_df['RID'].nunique()}\")\n\nprint(\"\\nREPEATED-CV COVERAGE SUMMARY\")\ndisplay(\n    cv_summary[\n        [\n            \"feature\",\n            \"quantile\",\n            \"mean_coverage_pct\",\n            \"median_coverage_pct\",\n            \"sd_coverage_pct\",\n            \"p10_coverage_pct\",\n            \"p90_coverage_pct\",\n        ]\n    ].round(2)\n)\n\nprint(\"\\nLOCKED TECHNICAL ENVELOPES\")\ndisplay(\n    thresholds[\n        [\n            \"feature\",\n            \"q90_pct\",\n            \"q90_boot_ci_low\",\n            \"q90_boot_ci_high\",\n            \"q95_pct\",\n            \"q95_boot_ci_low\",\n            \"q95_boot_ci_high\",\n        ]\n    ].round(3)\n)\n\nprint(\"\\nREPROCESSING SENSITIVITY\")\ndisplay(reproc_sensitivity.round(2))\n\nprint(\"\\nUNIQUE HARD-REVIEW ROOT CAUSES\")\ndisplay(root_cause_summary.round(1))\n\nprint(\"\\nPRIMARY FEATURE-LEVEL TRACE STATES — Q90 (%)\")\ndisplay(feature_state_pivot.round(1))\n\nprint(\"\\nSUBJECT PACKAGE STATE — DESCRIPTIVE\")\nprint(gate_df[\"TRACE_PACKAGE_STATE\"].value_counts())\nprint((100 * gate_df[\"TRACE_PACKAGE_STATE\"].value_counts(normalize=True)).round(1))\n\nprint(\"\\nPACKAGE RULE SENSITIVITY (%)\")\ndisplay(\n    package_sensitivity.pivot(\n        index=\"trusted_min_rule\",\n        columns=\"state\",\n        values=\"pct\"\n    ).fillna(0).round(1)\n)\n\nprint(\"\\nQ95 FEATURE-LEVEL SENSITIVITY (%)\")\ndisplay(\n    q95_sensitivity.pivot(\n        index=\"feature\",\n        columns=\"state\",\n        values=\"pct\"\n    ).fillna(0).round(1)\n)\n\nprint(\"\\nEXPLORATORY OUTCOME AUDIT — NOT USED FOR CALIBRATION\")\ndisplay(outcome_audit)\n\nprint(\"\\nREADINESS CHECKS\")\nchecks = {\n    \"real_same_day_repeat_reference\": len(acq) == 75,\n    \"subject_level_repeated_cv_completed\": len(cv) > 0,\n    \"bootstrap_threshold_uncertainty_completed\": thresholds[\"q90_boot_ci_low\"].notna().all(),\n    \"real_longitudinal_application\": gate_df[\"RID\"].nunique() == 258,\n    \"feature_level_three_state_output_present\":\n        set([\"TRUSTED\", \"CAUTION\", \"REVIEW_REQUIRED\"]).issubset(\n            set(feature_state_summary.loc[feature_state_summary[\"n\"] > 0, \"state\"])\n        ),\n    \"unique_root_cause_package_review\": True,\n    \"outcome_blind_gate_calibration\": True,\n}\n\nfor name, passed in checks.items():\n    print(f\"{name}: {'PASS' if passed else 'CHECK'}\")\n\nprint(\"\\nPRIMARY INTERPRETATION\")\nprint(\"Feature-level TRACE states are the primary gate output.\")\nprint(\"Subject-package state is secondary/descriptive.\")\nprint(\"Q90 is primary; Q95 is conservative sensitivity.\")\nprint(\"No clinical outcome was used to select thresholds or gate rules.\")\n",
 "# Re-create and download the private gated file for 21-REAL\n\nfrom google.colab import files\nfrom pathlib import Path\n\nFEATURES = [\n    \"LHIPPOC\",\n    \"RHIPPOC\",\n    \"LENTORHIN\",\n    \"RENTORHIN\",\n    \"LMIDTEMP\",\n    \"RMIDTEMP\",\n    \"VENTRICLES\",\n    \"ICV\",\n]\n\nBIO_FEATURES = [f for f in FEATURES if f != \"ICV\"]\n\nPRIVATE_GATE_PATH = Path(\n    \"/content/TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE.csv\"\n)\n\nprivate_cols = [\n    \"RID\",\n    \"LABEL\",\n    \"TRACE_PACKAGE_STATE\",\n    \"N_TRUSTED\",\n    \"N_CAUTION\",\n    \"N_REVIEW_FEATURES\",\n    \"N_UNIQUE_HARD_ROOT_CAUSES\",\n    \"ICV_HARD_REVIEW\",\n    \"TEMP_HARD_REVIEW\",\n    \"VENT_HARD_REVIEW\",\n    \"OVERALL_QC_BOTH_PASS\",\n]\n\nfor f in FEATURES:\n    for c in [\n        f\"BL_{f}\",\n        f\"M12_{f}\",\n        f\"{f}_ABS_LONG_PCT\",\n        f\"{f}_TECH_Q90\",\n        f\"{f}_TECH_Q95\",\n        f\"{f}_SIGNAL_TO_TECH_Q90\",\n    ]:\n        if c in gate_df.columns:\n            private_cols.append(c)\n\nfor f in BIO_FEATURES:\n    private_cols.append(f\"{f}_TRACE_STATE\")\n\nprivate_gate = gate_df[private_cols].copy()\n\nprivate_gate.to_csv(\n    PRIVATE_GATE_PATH,\n    index=False\n)\n\nprint(\"Saved:\")\nprint(PRIVATE_GATE_PATH)\nprint(\"Rows:\", len(private_gate))\nprint(\"Columns:\", len(private_gate.columns))\n\nfiles.download(str(PRIVATE_GATE_PATH))"
]
EXPECTED_ORIGINAL_CELL_SHA256 = [
 "9711bbeb64053d06a44ef0dc25b5262cba2293cc03d79ae71a8d0230d87e7e8c",
 "0dab9f54c64e42797d1d2f5958d0f8447b98486d8d8591f70032a13fcb0277db",
 "9ef09c9c5e9fee8eb2848322a259c4b2d7291656b6c9244ddaa540fb40bab676",
 "a8dd502e1ee574748404c67312ccc80fc806a80b05d335f5e04af906cc050cd4",
 "d06511ec3d32afadee8a4de7b321bf38cc2ae3b937a7040f670b77149bcd52f4",
 "9d08080a13a6e97f9051f560edb86959b38496bd228ab849fb6b2230c9b80308",
 "bdebb72c910598a8a908883ab2916e3bc738df567df13c03e486266ce5fcbd4d",
 "be79a4c290cc5610165d9db96eb71a15a0af636f08e28c1ca37f8a647bb10c8e",
 "742d012dbfbccadb9a3eca1a09f763258eb245c71b81a058751bb6d3bf363a85",
 "240b86fb62697a3b32dd75a2bc19bd99ba49d38e0bfbd020609d7ba12f4188ee",
 "537138b316bbe4410136f2b3f794f7c708ac89ab0cd96b5a5bb044346bf849a4",
 "472f835826730b7ab628eb2017be5e738d7a7102e86228a78ab876febba6bcf8",
 "f172cd5e8225f6238789025446d2a4179c1fb302da0032917f4538bb527ee51c",
 "d2550260d70a2b7a492393e10017f7b8bba09cf5db818899a64b2f7fc6030fc2",
 "1d9c3f145b805c4d8c1642eeaf2845934b976d99c8151db7babe406f39d80f86",
 "d06fc9dd9aa075c11aadcf07479adc64c5e455458e742d855b676b539286976e",
 "5a1d4dfd8ca4eaeb5a2fac759e68c3e7c674d139d8f81f9c5297bc8ac5865055",
 "48bcf4853c63e452c886ab290963257cea8375b798ec5e5720081440dd3b8668",
 "19862db6f8343cb68dbe94f4b98db1d81bf68b465a20e9a3121b8085cee8d0e9",
 "f1d23d0e42709184bad8efbed9c9f5a2efa4e5384d1003bc94bfc2820a9f93a8",
 "9326c3675794683deb95b07afbd0cab28cf247eabf41ec709f0f6850d3c7ad82"
]
PINNED_ORIGINAL_FINGERPRINT = '3e2ad0b25ab88aa0194dd08aa0efb8dd35b3c662be3e09c67ac89e2f4882f16f'
PINNED_ORIGIN_NOTEBOOK_SHA256 = 'abd5fd60cec1572db5c2e31948aa2a6626895b656d2b7cbce0fbb187061113cf'

assert len(ORIGINAL_20B_CELLS)==21
assert original_fingerprint(ORIGINAL_20B_CELLS)==PINNED_ORIGINAL_FINGERPRINT
assert all(hashlib.sha256(s.encode('utf-8')).hexdigest()==h for s,h in zip(ORIGINAL_20B_CELLS,EXPECTED_ORIGINAL_CELL_SHA256))
assert private15['selected_notebook']['sha256']==PINNED_ORIGIN_NOTEBOOK_SHA256
assert phase15['original_method']['selected_original_code_cell_count']==21
assert phase15['original_method']['selected_original_notebook_basename']==private15['selected_notebook']['filename']
assert not any('LABEL' in ORIGINAL_20B_CELLS[i] for i in range(3,12)), 'Original calibration unexpectedly uses clinical labels.'
for i,s in enumerate(ORIGINAL_20B_CELLS): ast.parse(s)
assert 'files.download(' in ORIGINAL_20B_CELLS[20]
assert not any('files.download(' in s for s in ORIGINAL_20B_CELLS[:19])
# The final original cell downloads a participant-level CSV, so it is intentionally excluded.
EXECUTE_ORIGINAL_CELL_INDICES=tuple(range(0,19))
copy_rel=private15.get('code_only_private_copy_path')
copy_path=contained_file(MYDRIVE,copy_rel)
private_copy_matches=None
if copy_path:
    saved=json.loads(copy_path.read_text(encoding='utf-8'))
    saved_sources=[src(c) for c in saved['cells'] if c.get('cell_type')=='code']
    private_copy_matches=(original_fingerprint(saved_sources)==PINNED_ORIGINAL_FINGERPRINT and
        saved.get('metadata',{}).get('ichi2027_origin_sha256')==PINNED_ORIGIN_NOTEBOOK_SHA256)
    assert private_copy_matches, 'Private 20B code copy changed since user-reviewed original. Stop and inspect.'
print({'exact_original_code_fingerprint_verified':True,'original_original_cells_pinned':21,
       'analytical_and_output_cells_to_run':len(EXECUTE_ORIGINAL_CELL_INDICES),
       'original_download_and_summary_cells_excluded':[19,20],
       'phase15_original_code_copy_matched':private_copy_matches})


{'exact_original_code_fingerprint_verified': True, 'original_original_cells_pinned': 21, 'analytical_and_output_cells_to_run': 19, 'original_download_and_summary_cells_excluded': [19, 20], 'phase15_original_code_copy_matched': True}


In [6]:
import pandas as pd
import numpy as np
FEATURES=['LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV']
LONG_NAME='TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'
REPEAT_NAME='TRACE_ADNI1_repeat_pairs_REAL.csv'
recs=private15['all_available_input_evidence_PRIVATE']
assert LONG_NAME in recs and REPEAT_NAME in recs, 'Phase15 did not verify both required originals.'
assert recs[LONG_NAME].get('phase09_source_sha_match') is True
assert recs[LONG_NAME].get('rows')==258 and recs[REPEAT_NAME].get('rows')==90
EXPECTED_INPUT_HASHES={n:recs[n]['sha256_PRIVATE'] for n in [LONG_NAME,REPEAT_NAME]}

def search_matching(name,expected):
    candidates=[]
    environment={'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv':'ICHI2027_LONG_CSV',
                 'TRACE_ADNI1_repeat_pairs_REAL.csv':'ICHI2027_REPEAT_CSV'}
    for s in (os.environ.get(environment[name]),recs[name].get('path_PRIVATE'),
              str(PROJECT/'source_inputs'/name),str(PROJECT/'data'/name),
              str(PROJECT/'private_inputs'/name),str(Path('/content')/name)):
        if s:candidates.append(Path(s))
    for p in candidates:
        if p.is_file() and sha256_file(p)==expected:return p
    # Targeted filename search. Do not accept a same-named but altered input.
    for current,dirs,names in os.walk(MYDRIVE):
        dirs[:]=[d for d in dirs if d not in {'.git','.ipynb_checkpoints','__pycache__','node_modules'}]
        if name in names:
            p=Path(current)/name
            if sha256_file(p)==expected:return p
    return None

selected_inputs={n:search_matching(n,h) for n,h in EXPECTED_INPUT_HASHES.items()}
missing=[n for n,p in selected_inputs.items() if p is None]
if missing and os.environ.get('ICHI2027_PROMPT_UPLOAD','1')=='1':
    try:
        from google.colab import files
        print('Upload ONLY the original unchanged file(s):',missing)
        uploaded=files.upload()
        for n in missing:
            if n in uploaded:
                p=RUNROOT/n;p.write_bytes(uploaded[n])
                if sha256_file(p)!=EXPECTED_INPUT_HASHES[n]:
                    raise ValueError(n+': uploaded bytes differ from verified Phase15 original.')
                selected_inputs[n]=p
    except ImportError:pass
assert all(selected_inputs.values()),'Both exact Phase15-verified CSV inputs are required before replay.'
assert all(sha256_file(p)==EXPECTED_INPUT_HASHES[n] for n,p in selected_inputs.items())
assert phase15['authentic_input_preflight']['phase09_longitudinal_digest_comparison']=='MATCH'
print('Both verified real inputs located and matched to the original Phase15 private hashes.')


Both verified real inputs located and matched to the original Phase15 private hashes.


In [7]:
long_check=pd.read_csv(selected_inputs[LONG_NAME],low_memory=False)
rep_check=pd.read_csv(selected_inputs[REPEAT_NAME],low_memory=False)
required_long=({'RID','LABEL','BL_EXAMDATE','M12_EXAMDATE','BL_OVERALLQC','M12_OVERALLQC',
                'BL_TEMPQC','M12_TEMPQC','BL_VENTQC','M12_VENTQC'}|
               {f'{v}_{f}' for v in ('BL','M12') for f in FEATURES})
required_repeat=({'RID','REPEAT_TYPE','EXAMDATE','IMAGEUID_1','IMAGEUID_2',
                  'LONIUID_1','LONIUID_2','QC_1','QC_2','TEMPQC_1','TEMPQC_2',
                  'VENTQC_1','VENTQC_2'}|
                 {f'PCT_DIFF_{f}' for f in FEATURES}|
                 {f'{f}_{n}' for f in FEATURES for n in (1,2)})
assert required_long.issubset(long_check.columns), 'Longitudinal original-method input columns missing.'
assert required_repeat.issubset(rep_check.columns), 'Repeat-pair original-method input columns missing.'
assert len(long_check)==258 and len(long_check.columns)==51
assert len(rep_check)==90 and len(rep_check.columns)==51
assert long_check['RID'].notna().all() and long_check['RID'].is_unique
assert rep_check['RID'].notna().all()
assert rep_check['RID'].isin(long_check['RID']).sum()>=1
acq_check=rep_check.loc[rep_check['REPEAT_TYPE']=='SAME_DAY_DIFFERENT_ACQUISITION'].copy()
reproc_check=rep_check.loc[rep_check['REPEAT_TYPE']=='SAME_SOURCE_REPROCESSING'].copy()
assert len(acq_check)==75 and len(reproc_check)==15, 'Unexpected repeat-pair subgroups; do not silently recalibrate.'
assert acq_check['RID'].nunique()>=5
problems={}
for f in FEATURES:
    for v in ('BL','M12'):
        col=f'{v}_{f}'
        numeric=pd.to_numeric(long_check[col],errors='coerce')
        problems[col+'_nonfinite']=int((~np.isfinite(numeric)).sum())
        problems[col+'_nonpositive']=int((numeric<=0).sum())
    for g,label in ((acq_check,'acquisition'),(reproc_check,'reprocessing')):
        numeric=pd.to_numeric(g[f'PCT_DIFF_{f}'],errors='coerce')
        problems[f'{label}_{f}_nonfinite']=int((~np.isfinite(numeric)).sum())
        problems[f'{label}_{f}_negative']=int((numeric<0).sum())
    for k in (1,2):
        numeric=pd.to_numeric(rep_check[f'{f}_{k}'],errors='coerce')
        problems[f'repeat_{f}_{k}_nonfinite']=int((~np.isfinite(numeric)).sum())
        problems[f'repeat_{f}_{k}_nonpositive']=int((numeric<=0).sum())
assert all(v==0 for v in problems.values()),'Missing, nonpositive or invalid MRI input detected; inspect privately.'
assert long_check['BL_EXAMDATE'].notna().all() and long_check['M12_EXAMDATE'].notna().all()
assert set(acq_check['REPEAT_TYPE'].unique())=={'SAME_DAY_DIFFERENT_ACQUISITION'}
source_pair_independently_verified=False
same_loni_pairs=int((rep_check['LONIUID_1'].astype(str)==rep_check['LONIUID_2'].astype(str)).sum())
# Only one EXAMDATE per repeat-pair record: genuine distinct acquisition dates remain unconfirmed.
preflight={'rows_longitudinal':len(long_check),'rows_repeat_pairs':len(rep_check),
           'source_numeric_events':int(2*len(FEATURES)*len(long_check)),
           'acquisition_pairs_source_labeled':len(acq_check),
           'reprocessing_pairs_source_labeled':len(reproc_check),
           'same_loniuid_repeat_pairs':same_loni_pairs,
           'two_acquisition_dates_independently_verified':source_pair_independently_verified,
           'original_input_checks_passed':True}
print(preflight)


{'rows_longitudinal': 258, 'rows_repeat_pairs': 90, 'source_numeric_events': 4128, 'acquisition_pairs_source_labeled': 75, 'reprocessing_pairs_source_labeled': 15, 'same_loniuid_repeat_pairs': 16, 'two_acquisition_dates_independently_verified': False, 'original_input_checks_passed': True}


In [8]:
# Original unmodified data-loading cell prioritizes /content. Stage only checksum-verified exact bytes.
# Preserve any pre-existing same-name /content file inside this isolated temporary directory.
previous_content_backups=[]
for name,p in selected_inputs.items():
    target=Path('/content')/name if Path('/content').is_dir() else RUNROOT/name
    if target.exists() and sha256_file(target)!=EXPECTED_INPUT_HASHES[name]:
        backup=RUNROOT/('previous_content_'+name)
        shutil.copyfile(target,backup)
        previous_content_backups.append(name)
    if p.resolve()!=target.resolve():shutil.copyfile(p,target)
    assert sha256_file(target)==EXPECTED_INPUT_HASHES[name]
    # The original code selects /content or CWD, so verify the actual path it will select.
    if Path('/content').is_dir(): assert target==Path('/content')/name
os.chdir(RUNROOT)
original_code_execution={'attempted':False,'completed':False,'original_cells_executed':[],
                         'failure_original_cell_index':None,'failure_type':None}
print('Isolated original 20B replay prepared; existing /content name collisions backed up:',len(previous_content_backups))


Isolated original 20B replay prepared; existing /content name collisions backed up: 0


In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
orig_namespace={'__builtins__':__builtins__,'__name__':'__main__','display':lambda *args,**kwargs:None}
replay_log=PRIVATE/'ICHI2027_Phase16_Original_20B_Runtime_Log_PRIVATE.txt'
original_code_execution['attempted']=True
with replay_log.open('w',encoding='utf-8') as log:
    try:
        with contextlib.redirect_stdout(log),contextlib.redirect_stderr(log):
            for i in EXECUTE_ORIGINAL_CELL_INDICES:
                # Exact source from the uploaded original user-reviewed notebook, byte-hash pinned.
                assert hashlib.sha256(ORIGINAL_20B_CELLS[i].encode()).hexdigest()==EXPECTED_ORIGINAL_CELL_SHA256[i]
                exec(compile(ORIGINAL_20B_CELLS[i],f'original_20B_cell_{i}.py','exec'),orig_namespace)
                original_code_execution['original_cells_executed'].append(i)
                if i==0:
                    # Only suppress display; all scientific computations and original writes are untouched.
                    orig_namespace['plt'].show=lambda *args,**kwargs:None
                if i==1:
                    assert sha256_file(orig_namespace['LONGITUDINAL_PATH'])==EXPECTED_INPUT_HASHES[LONG_NAME]
                    assert sha256_file(orig_namespace['REPEAT_PATH'])==EXPECTED_INPUT_HASHES[REPEAT_NAME]
                if i==4:
                    assert len(orig_namespace['thresholds'])==8
                    assert orig_namespace['thresholds']['q90_pct'].notna().all()
        original_code_execution['completed']=True
    except Exception as exc:
        original_code_execution['failure_original_cell_index']=i
        original_code_execution['failure_type']=type(exc).__name__
        failure={'phase':'16_controlled_original_20b_real_replay',
                 'execution':original_code_execution,'no_four_arm_scores':True,
                 'privacy':'No participant-level data exported.'}
        SHAREABLE.write_text(json.dumps(failure,indent=2),encoding='utf-8')
        print('Original cell failed; safe status JSON was saved. Review PRIVATE runtime log on Drive.')
        raise RuntimeError(f'Original method failed at cell {i}: {type(exc).__name__}; inspect PRIVATE log') from None
assert original_code_execution['completed']
assert original_code_execution['original_cells_executed']==list(range(19))
print('Original corrected 20B cells 0–18 executed with exact, unchanged scientific code. Cells 19 and 20 intentionally skipped.')


Original corrected 20B cells 0–18 executed with exact, unchanged scientific code. Cells 19 and 20 intentionally skipped.


In [10]:
from importlib.metadata import version,PackageNotFoundError
m=orig_namespace
assert sha256_file(m['LONGITUDINAL_PATH'])==EXPECTED_INPUT_HASHES[LONG_NAME]
assert sha256_file(m['REPEAT_PATH'])==EXPECTED_INPUT_HASHES[REPEAT_NAME]
assert len(m['long_df'])==258 and len(m['rep_df'])==90
assert len(m['acq'])==75 and len(m['reproc'])==15
assert len(m['gate_df'])==258 and m['gate_df']['RID'].is_unique
assert len(m['thresholds'])==8 and len(m['feature_state_summary'])==21
assert len(m['cv'])==25*5*8*2
assert len(m['private_gate'])==258
for f in FEATURES:
    assert np.isfinite(m['thresholds'].loc[m['thresholds']['feature']==f,['q90_pct','q95_pct','q90_boot_ci_low','q90_boot_ci_high','q95_boot_ci_low','q95_boot_ci_high']].to_numpy(dtype=float)).all()
    row=m['thresholds'].loc[m['thresholds']['feature']==f].iloc[0]
    assert row['q90_pct']<=row['q95_pct']
    if f!='ICV':
        assert m['gate_df'][f'{f}_TRACE_STATE'].isin(['TRUSTED','CAUTION','REVIEW_REQUIRED']).all()
assert m['gate_df']['TRACE_PACKAGE_STATE'].isin(['TRUSTED','CAUTION','REVIEW_REQUIRED']).all()
for f in ['table_repeated_cv_all_folds.csv','table_locked_thresholds_bootstrap.csv',
          'table_feature_trace_states_q90.csv','manifest.json']:
    assert (RUNROOT/'trace_20b_real_corrected_outputs'/f).is_file()
assert (RUNROOT/'TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE.csv').is_file()
assert int((m['gate_df'][[f'{f}_TRACE_STATE' for f in FEATURES if f!='ICV']].notna()).sum().sum())==258*7
# Original private CSV kept its participant-level RID; it must remain private.
OUTPUT_COPY=PRIVATE/'original_20b_replayed_outputs'
OUTPUT_COPY.mkdir(exist_ok=True)
for p in (RUNROOT/'trace_20b_real_corrected_outputs').iterdir():
    if p.is_file() and p.suffix.lower() in {'.csv','.json','.png'}:
        shutil.copy2(p,OUTPUT_COPY/p.name)
shutil.copy2(RUNROOT/'TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE.csv',
             PRIVATE/'TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE_REGENERATED.csv')
calib_sha=sha256_file(OUTPUT_COPY/'table_locked_thresholds_bootstrap.csv')
PRIVATE_SIDECAR=PRIVATE/'ICHI2027_Phase16_TRACE_LONGITUDINAL_JOIN_SIDECAR_PRIVATE.csv'
join_cols=['RID','BL_EXAMDATE','M12_EXAMDATE','BL_IMAGEUID','M12_IMAGEUID',
           'BL_LONIUID','M12_LONIUID']
assert set(join_cols).issubset(m['gate_df'].columns)
sidecar=m['gate_df'][join_cols+['TRACE_PACKAGE_STATE','N_UNIQUE_HARD_ROOT_CAUSES']+
    [f'{f}_TRACE_STATE' for f in FEATURES if f!='ICV']].copy()
sidecar['TRACE_RULE_VERSION']='20B_REAL_CORRECTED_Q90_REPLAY_PHASE16'
sidecar['TRACE_ORIGINAL_SOURCE_NOTEBOOK_SHA256']=PINNED_ORIGIN_NOTEBOOK_SHA256
sidecar['TRACE_REPLAY_CALIBRATION_CSV_SHA256']=calib_sha
sidecar['TRACE_STATE_GRANULARITY']='SUBJECT_PAIR_BL_TO_M12_NOT_INDIVIDUAL_SCAN'
assert len(sidecar)==258 and sidecar['RID'].is_unique
sidecar.to_csv(PRIVATE_SIDECAR,index=False)
assert sha256_file(PRIVATE_SIDECAR)
print({'original_20b_replay_passed':True,'participant_level_data_saved_only_in_private_drive':True,
       'private_longitudinal_pair_sidecar_created':True,'participant_join_records':len(sidecar),
       'calibration_outputs_saved_private':True})


{'original_20b_replay_passed': True, 'participant_level_data_saved_only_in_private_drive': True, 'private_longitudinal_pair_sidecar_created': True, 'participant_join_records': 258, 'calibration_outputs_saved_private': True}


In [11]:
def safe_count(n):
    n=int(n)
    return n if n>=5 else '<5'

def version_or_none(n):
    try:return version(n)
    except PackageNotFoundError:return None

states=['TRUSTED','CAUTION','REVIEW_REQUIRED']
feature_state_public={}
for f in [x for x in FEATURES if x!='ICV']:
    counts=m['gate_df'][f'{f}_TRACE_STATE'].value_counts()
    feature_state_public[f]={s:safe_count(counts.get(s,0)) for s in states}
threshold_public={}
for r in m['thresholds'].to_dict('records'):
    threshold_public[r['feature']]={
        'calibration_pair_count':int(r['n_pairs']),
        'calibration_subject_count':int(r['n_subjects']),
        'q90_pct':float(r['q90_pct']),
        'q90_95pct_cluster_bootstrap_CI':[float(r['q90_boot_ci_low']),float(r['q90_boot_ci_high'])],
        'q95_pct':float(r['q95_pct']),
        'q95_95pct_cluster_bootstrap_CI':[float(r['q95_boot_ci_low']),float(r['q95_boot_ci_high'])]
    }
package_counts=m['gate_df']['TRACE_PACKAGE_STATE'].value_counts()
report={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'16_controlled_original_20b_real_replay',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'status':'ACTUAL_ORIGINAL_20B_METHOD_REEXECUTED_ON_PHASE15_HASH_VERIFIED_REAL_INPUTS_NO_MICDM_LOAD_OR_FOUR_ARM_COMPARISON',
 'privacy':'Aggregate counts suppressed when <5; no RID, dates, image identifiers, source file hashes, raw CSV paths or participant-level TRACE outputs.',
 'phase15_shareable_sha256':sha256_file(phase15_path),
 'original_method':{
     'origin_notebook_sha256':PINNED_ORIGIN_NOTEBOOK_SHA256,
     'original_code_fingerprint':PINNED_ORIGINAL_FINGERPRINT,
     'original_code_only_copy_matched':private_copy_matches,
     'executed_exact_original_cell_indices':original_code_execution['original_cells_executed'],
     'excluded_original_cell_indices':[19,20],
     'method_changed':False,
     'random_seed':2027,
     'primary_quantile':90,'sensitivity_quantile':95,
     'repeated_subject_group_cv_repeats':25,
     'repeated_subject_group_cv_folds':5,
     'subject_cluster_bootstrap_replicates_per_feature_and_quantile':2000,
     'no_synthetic_participant_data_created':True,
     'original_20b_frozen_historical_numerical_outputs_available_for_exact_match':False
 },
 'verified_replay_inputs':{
    'longitudinal_rows':len(m['long_df']),
    'repeat_pair_rows':len(m['rep_df']),
    'repeat_acquisition_pairs_source_labeled':len(m['acq']),
    'repeat_reprocessing_pairs_source_labeled':len(m['reproc']),
    'repeat_acquisition_subjects':int(m['acq']['RID'].nunique()),
    'repeat_reprocessing_subjects':int(m['reproc']['RID'].nunique()),
    'longitudinal_subjects':int(m['gate_df']['RID'].nunique()),
    'repeat_pair_same_loniuid_rows':same_loni_pairs,
    'repeat_pair_two_acquisition_dates_independently_verified':False,
    'both_inputs_sha256_equal_phase15_private_manifest':True,
    'longitudinal_sha256_equal_prior_phase09_private_manifest_via_phase15':True
 },
 'actual_calibration':{
    'repeated_cv_folds_evaluated':len(m['cv']),
    'eight_source_features_calibrated':len(m['thresholds']),
    'thresholds_by_feature':threshold_public,
    'all_bootstrap_intervals_finite':True
 },
 'actual_regenerated_trace':{
    'feature_level_reliability_counts_suppress_lt5':feature_state_public,
    'subject_pair_package_state_counts_suppress_lt5':{s:safe_count(package_counts.get(s,0)) for s in states},
    'participant_pair_rows_private':len(sidecar),
    'biological_feature_states_private':int(len(sidecar)*7),
    'sidecar_granularity':'BL_TO_M12_SUBJECT_PAIR_NOT_SCAN_LEVEL',
    'reference':'NEW_REGENERATED_OUTPUT_OF_EXACT_ORIGINAL_20B_CODE_NOT_HISTORICAL_SIDECAR_RECOVERY',
    'private_artifact_filenames':['TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE_REGENERATED.csv',
          'ICHI2027_Phase16_TRACE_LONGITUDINAL_JOIN_SIDECAR_PRIVATE.csv',
          'original_20b_replayed_outputs/table_locked_thresholds_bootstrap.csv']
 },
 'environment':{'python':sys.version.split()[0],'numpy':np.__version__,
    'pandas':pd.__version__,'matplotlib':matplotlib.__version__},
 'limitations':{
    'original_historical_frozen_numeric_results_recovered':False,
    'source_repeat_pair_type_independently_verified_from_original_ADNI_acquisition_records':False,
    'exact_local_ADNI_export_units_verified':False,
    'DICOM_StudyInstanceUID_and_SeriesInstanceUID_verified':False,
    'full_athena_snapshot_loaded':False,
    'OMOP_concept_mappings_approved':False,
    'original_generic_TFL_v15_on_real_MRI_executed':False,
    'official_MI_CDM_load_completed':False,
    'genuine_four_arm_comparison_completed':False
 },
 'four_arm_comparator_scores':None,'official_micdm_load_results':None,
 'next_gate':'REUSE_REAL_REGENERATED_TRACE_PAIR_SIDECAR_IN_AUDITABLE_FHIR_LINKAGE_WITHOUT_PRETENDING_SCAN_LEVEL_OR_OFFICIAL_MICDM; STILL_OBTAIN_ORIGINAL_ADNI_EXPORT_AND_ATHENA'
}
PRIVATE_REPORT=PRIVATE/'ICHI2027_Phase16_PRIVATE_REPLAY_MANIFEST.json'
private_report={'run_utc':report['run_utc'],
 'phase15_private_sha256':sha256_file(private15_path),
 'phase15_verified_original_input_hashes_PRIVATE':EXPECTED_INPUT_HASHES,
 'source_input_paths_PRIVATE':{n:str(p) for n,p in selected_inputs.items()},
 'original_notebook_origin_sha256':PINNED_ORIGIN_NOTEBOOK_SHA256,
 'original_code_fingerprint':PINNED_ORIGINAL_FINGERPRINT,
 'original_cell_source_hashes':EXPECTED_ORIGINAL_CELL_SHA256,
 'original_runtime_artifacts_PRIVATE':str(OUTPUT_COPY),
 'original_private_gate_output_PRIVATE':str(PRIVATE/'TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE_REGENERATED.csv'),
 'new_private_pair_sidecar':str(PRIVATE_SIDECAR),
 'private_pair_sidecar_sha256':sha256_file(PRIVATE_SIDECAR),
 'original_calibration_csv_sha256':calib_sha,
 'same_day_pair_verification':'SOURCE_LABEL_ONLY',
 'original_historical_output_comparison':'NOT_AVAILABLE'}
PRIVATE_REPORT.write_text(json.dumps(private_report,indent=2,ensure_ascii=False),encoding='utf-8')
SHAREABLE.write_text(json.dumps(report,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('Phase16 aggregate-only report saved to:',SHAREABLE)
print('Original regenerated TRACE and detailed private manifest saved to:',PRIVATE)


Phase16 aggregate-only report saved to: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase16_outputs/ICHI2027_Phase16_SHAREABLE.json
Original regenerated TRACE and detailed private manifest saved to: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase16_outputs/_PRIVATE_DO_NOT_SHARE


In [12]:
try:
    from google.colab import files
    files.download(str(SHAREABLE))
except ImportError:
    print(SHAREABLE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>